[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WesleyInfoBr2/cienciadedados/blob/main/capitulos/cap09_series_temporais/modelagem_classica_validacao_temporal_autoveiculos.ipynb)

# Modelagem clássica, diagnóstico e validação temporal

Este notebook usa a base SériesTemporais_Autoveículos para explorar FAC e FACP, avaliar transformações, estimar um modelo ARIMA ou sazonal quando justificável, diagnosticar resíduos e comparar previsões com uma referência ingênua. A avaliação é feita em período fora do treinamento, com MAE e RMSE.

O botão pressupõe publicação do arquivo em `capitulos/cap09_series_temporais/modelagem_classica_validacao_temporal_autoveiculos.ipynb` no repositório do livro.

## 1. Objetivo e protocolo de validação

A modelagem clássica de séries temporais exige preservar a ordem cronológica. O período final de teste deve ser separado antes da escolha da transformação, das ordens do modelo e de outras decisões. O treinamento é usado para exploração e estimação; o teste é usado apenas ao final para comparar as previsões com os valores observados.

FAC e FACP ajudam a formular candidatos, mas não determinam mecanicamente as ordens. A escolha deve considerar a estacionariedade, a sazonalidade, a parcimônia, o diagnóstico dos resíduos e o desempenho fora do treinamento. O exemplo usa previsões de horizonte fixo para todo o período de teste, sem reestimar o modelo com observações do teste.

In [ ]:
import io
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox
from sklearn.metrics import mean_absolute_error, mean_squared_error

SEMENTE=2026
URL_BASE="https://github.com/WesleyInfoBr2/cienciadedados/raw/refs/heads/main/dados/S%C3%A9riesTemporais_Autove%C3%ADculos.xlsx"
np.random.seed(SEMENTE)

## 2. Carregamento e inspeção

A planilha é carregada do repositório do projeto. Inspecione as abas, colunas e registros antes de selecionar a medida e a categoria que comporão a série. O notebook não presume os nomes das colunas.

In [ ]:
def carregar_excel():
    try:
        with urllib.request.urlopen(URL_BASE,timeout=30) as resposta:
            return pd.ExcelFile(io.BytesIO(resposta.read()))
    except Exception as erro:
        print("Download indisponível:",erro)
        try:
            from google.colab import files
            enviados=files.upload()
            if not enviados: raise RuntimeError("Nenhum arquivo selecionado.")
            return pd.ExcelFile(io.BytesIO(enviados[next(iter(enviados))]))
        except ImportError:
            raise RuntimeError("Execute no Colab ou disponibilize a planilha localmente.")

excel=carregar_excel()
bases={aba:pd.read_excel(excel,sheet_name=aba) for aba in excel.sheet_names}
for aba,df in bases.items():
    print(f"\n{aba}: {df.shape}")
    print(list(df.columns))
    display(df.head())

## 3. Seleção e preparação da série mensal

Configure os nomes exatos da aba, data e medida. Se necessário, filtre uma categoria ou grupo. Informe a agregação mensal de acordo com o significado da variável: soma para fluxo em unidades compatíveis, média para taxas ou outra regra justificada. Verifique registros repetidos por mês, continuidade, valores ausentes e sinais da medida. Não complete lacunas automaticamente.

In [ ]:
ABA=None
COLUNA_DATA=None
COLUNA_VALOR=None
COLUNA_CATEGORIA=None
FILTRO_CATEGORIA=None
AGREGACAO="sum"  # 'sum' ou 'mean', escolhida conforme a natureza da medida

if not all([ABA,COLUNA_DATA,COLUNA_VALOR]) or ABA not in bases:
    raise ValueError("Configure ABA, COLUNA_DATA e COLUNA_VALOR após inspecionar a planilha.")
df=bases[ABA].copy()
if COLUNA_CATEGORIA and FILTRO_CATEGORIA:
    df=df[df[COLUNA_CATEGORIA].astype(str).str.contains(str(FILTRO_CATEGORIA),case=False,na=False,regex=False)].copy()
if COLUNA_DATA not in df.columns or COLUNA_VALOR not in df.columns:
    raise KeyError("Confira os nomes das colunas configuradas.")
df[COLUNA_DATA]=pd.to_datetime(df[COLUNA_DATA],errors="coerce",dayfirst=True)
df[COLUNA_VALOR]=pd.to_numeric(df[COLUNA_VALOR],errors="coerce")
df=df.dropna(subset=[COLUNA_DATA,COLUNA_VALOR]).copy()
if AGREGACAO not in ["sum","mean"]: raise ValueError("AGREGACAO deve ser 'sum' ou 'mean'.")
df["mes"]=df[COLUNA_DATA].dt.to_period("M")
print("Registros por mês, resumo:")
display(df.groupby("mes").size().describe().to_frame("n_registros"))
serie=df.groupby("mes")[COLUNA_VALOR].agg(AGREGACAO).sort_index()
indice=pd.period_range(serie.index.min(),serie.index.max(),freq="M")
serie=serie.reindex(indice)
serie.index=serie.index.to_timestamp(how="start")
print("Observações mensais:",len(serie),"| ausentes:",int(serie.isna().sum()))
print("Período:",serie.index.min(),"a",serie.index.max())
if serie.isna().any(): raise ValueError("Investigue meses ausentes antes da modelagem.")
if len(serie)<48: raise ValueError("Recomenda-se intervalo mais longo para avaliar sazonalidade anual e reservar teste.")
if (serie<=0).any(): print("Há valores não positivos: transformação logarítmica não será usada sem justificativa.")
serie.plot(figsize=(12,4),title="Série mensal selecionada");plt.ylabel(COLUNA_VALOR);plt.show()

## 4. Separação temporal antes das escolhas

Reserva-se o período final antes de examinar FAC/FACP, escolher transformações ou estimar modelos. Por padrão, são reservados os últimos 12 meses. O horizonte deve ser coerente com a aplicação; se houver decisão diferente, registre sua justificativa sem usar os resultados do teste para orientar as escolhas.

In [ ]:
HORIZONTE_TESTE=12
if len(serie)-HORIZONTE_TESTE<36:
    raise ValueError("Treinamento curto após a reserva. Ajuste o período com justificativa.")
treino=serie.iloc[:-HORIZONTE_TESTE].copy()
teste=serie.iloc[-HORIZONTE_TESTE:].copy()
print("Treinamento:",treino.index.min().date(),"a",treino.index.max().date(),len(treino))
print("Teste reservado:",teste.index.min().date(),"a",teste.index.max().date(),len(teste))
fig,ax=plt.subplots(figsize=(12,4));ax.plot(treino,label="Treinamento");ax.axvspan(teste.index.min(),teste.index.max(),alpha=.18,color="orange",label="Teste")
ax.set_title("Corte temporal definido antes da modelagem");ax.legend();plt.show()

## 5. Transformações e estacionariedade

Transformações podem ser consideradas para estabilizar variância ou tornar a série mais adequada à modelagem. O logaritmo exige valores estritamente positivos. A diferenciação regular remove mudanças de nível persistentes; a diferenciação sazonal pode ser considerada para periodicidade anual em dados mensais. As decisões são feitas somente com o treinamento.

ADF e KPSS são diagnósticos complementares com hipóteses nulas diferentes. Os resultados devem ser considerados em conjunto com os gráficos, FAC/FACP e conhecimento do processo, não como regra automática para definir o número de diferenças.

In [ ]:
def relatar_testes(x,nome):
    print("\n",nome)
    try:
        a=adfuller(x,autolag="AIC")
        print("ADF: estatística=",round(a[0],3),"p=",round(a[1],4),"lags=",a[2])
    except Exception as e: print("ADF indisponível:",e)
    try:
        k=kpss(x,regression="c",nlags="auto")
        print("KPSS: estatística=",round(k[0],3),"p=",round(k[1],4),"lags=",k[2])
    except Exception as e: print("KPSS indisponível:",e)

relatar_testes(treino,"Nível original, treinamento")
if (treino>0).all():
    treino_log=np.log(treino)
    treino_modelagem=treino_log
    NOME_TRANSFORMACAO="log"
    print("Log disponível; examine se ajuda a estabilizar a variância.")
    fig,ax=plt.subplots(1,2,figsize=(12,4));ax[0].plot(treino);ax[0].set_title("Nível original");ax[1].plot(treino_log);ax[1].set_title("Log")
    plt.tight_layout();plt.show()
else:
    treino_modelagem=treino.copy()
    NOME_TRANSFORMACAO="nenhuma"
    print("Log omitido devido a valores não positivos.")

# Diagnóstico de estacionariedade e FAC/FACP após uma diferença regular.
d1=treino_modelagem.diff().dropna()
relatar_testes(d1,"Primeira diferença do treinamento (exploratória)")
fig,ax=plt.subplots(1,2,figsize=(12,4))
plot_acf(d1,lags=min(36,len(d1)//2-1),zero=False,ax=ax[0]);ax[0].set_title("FAC da primeira diferença")
plot_pacf(d1,lags=min(36,len(d1)//2-1),zero=False,method="ywm",ax=ax[1]);ax[1].set_title("FACP da primeira diferença")
plt.tight_layout();plt.show()

## 6. FAC, FACP e identificação de candidatos

FAC e FACP da série transformada e diferenciada auxiliam a identificar dependências temporais e a formular ordens candidatas para os termos autorregressivos e de médias móveis. Padrões empíricos podem ser ambíguos, especialmente com amostras limitadas, sazonalidade ou mudanças estruturais.

Para fins didáticos, a célula seguinte permite especificar um pequeno conjunto de candidatos ARIMA. Se a periodicidade anual e os padrões sazonais justificarem, também é possível incluir um candidato SARIMA com período 12. A lista deve ser definida usando apenas o treinamento e registrada antes da avaliação no teste.

In [ ]:
# Edite os candidatos após examinar FAC/FACP e justificar ordens.
# ARIMA: (p,d,q). SARIMA: order=(p,d,q), seasonal_order=(P,D,Q,12).
CANDIDATOS=[
    {"nome":"ARIMA(1,1,0)","order":(1,1,0),"seasonal_order":(0,0,0,0)},
    {"nome":"ARIMA(0,1,1)","order":(0,1,1),"seasonal_order":(0,0,0,0)},
    {"nome":"ARIMA(1,1,1)","order":(1,1,1),"seasonal_order":(0,0,0,0)}
]
# Inclua SARIMA somente se houver justificativa sazonal, por exemplo:
# CANDIDATOS.append({"nome":"SARIMA(...) sazonal 12", "order":(p,d,q), "seasonal_order":(P,D,Q,12)})
if not CANDIDATOS: raise ValueError("Defina candidatos ARIMA justificados com base no treinamento.")
print("Candidatos a estimar:")
for c in CANDIDATOS: print(c)

## 7. Estimação e seleção no treinamento

Os candidatos são estimados somente no treinamento. AIC e BIC são apresentados como critérios auxiliares de comparação, não como substitutos do diagnóstico residual ou da validação temporal. Para uma transformação logarítmica, as previsões são reconvertidas à escala original por exponenciação; essa transformação simples da previsão média pode não coincidir com a média condicional na escala original, portanto a interpretação deve ser cautelosa.

In [ ]:
ajustes=[]
for c in CANDIDATOS:
    try:
        modelo=ARIMA(treino_modelagem,order=c["order"],seasonal_order=c["seasonal_order"],
                     enforce_stationarity=False,enforce_invertibility=False)
        ajuste=modelo.fit()
        ajustes.append({"nome":c["nome"],"config":c,"ajuste":ajuste,
                        "AIC":ajuste.aic,"BIC":ajuste.bic})
        print(c["nome"],"AIC",round(ajuste.aic,2),"BIC",round(ajuste.bic,2))
    except Exception as erro:
        print("Falha em",c["nome"],":",erro)
if not ajustes: raise RuntimeError("Nenhum candidato foi estimado com sucesso.")
display(pd.DataFrame([{k:v for k,v in a.items() if k in ["nome","AIC","BIC"]} for a in ajustes]).set_index("nome").sort_values("AIC"))

# Escolha explícita com base em parcimônia, diagnósticos e critérios auxiliares.
NOME_MODELO_ESCOLHIDO=None
if NOME_MODELO_ESCOLHIDO is None or NOME_MODELO_ESCOLHIDO not in [a["nome"] for a in ajustes]:
    raise ValueError("Defina NOME_MODELO_ESCOLHIDO entre os modelos estimados, após justificar a decisão.")
selecionado=next(a for a in ajustes if a["nome"]==NOME_MODELO_ESCOLHIDO)
resultado=selecionado["ajuste"]
print(resultado.summary())

## 8. Diagnóstico dos resíduos

Um modelo adequado deve deixar resíduos sem padrão temporal sistemático. São examinados gráfico temporal, distribuição, FAC residual e teste de Ljung-Box. A hipótese nula do Ljung-Box é ausência de autocorrelação conjunta até as defasagens avaliadas; um resultado não significativo não prova que o modelo seja correto. Examine também convergência e avisos da estimação.

In [ ]:
residuos=pd.Series(resultado.resid,index=treino_modelagem.index).dropna()
fig,ax=plt.subplots(2,2,figsize=(12,8))
ax[0,0].plot(residuos);ax[0,0].set_title("Resíduos no treinamento")
ax[0,1].hist(residuos,bins=20);ax[0,1].set_title("Distribuição dos resíduos")
plot_acf(residuos,lags=min(36,len(residuos)//2-1),zero=False,ax=ax[1,0]);ax[1,0].set_title("FAC dos resíduos")
from scipy import stats
stats.probplot(residuos,dist="norm",plot=ax[1,1]);ax[1,1].set_title("Gráfico Q-Q")
plt.tight_layout();plt.show()

lags=min(12,max(1,len(residuos)//5))
try:
    lb=acorr_ljungbox(residuos,lags=[lags],return_df=True,model_df=sum(selecionado["config"]["order"])+sum(selecionado["config"]["seasonal_order"][:3]))
    display(lb)
except Exception as erro: print("Ljung-Box indisponível:",erro)

## 9. Referência ingênua e avaliação fora do treinamento

A referência ingênua prevê, para todos os meses do horizonte, o último valor observado no treinamento. Essa referência simples permite verificar se o modelo oferece melhoria em relação a uma regra básica. O modelo ARIMA/SARIMA escolhido é ajustado no treinamento e produz previsões para o horizonte reservado. MAE e RMSE são calculados na escala original da série.

Não há reestimação com os dados de teste neste protocolo. Uma validação rolling-origin poderia oferecer avaliação mais abrangente, mas exigiria um desenho adicional e decisões prévias sobre janela e horizonte.

In [ ]:
h=len(teste)
previsao_modelo=resultado.get_forecast(steps=h).predicted_mean
if NOME_TRANSFORMACAO=="log":
    previsao_modelo=np.exp(previsao_modelo)
previsao_modelo=pd.Series(np.asarray(previsao_modelo),index=teste.index,name="ARIMA/SARIMA")
previsao_ingenua=pd.Series(float(treino.iloc[-1]),index=teste.index,name="Ingênua: último valor")

comparacao=pd.DataFrame({"observado":teste,"ingenua":previsao_ingenua,"modelo":previsao_modelo})
mae_ing=mean_absolute_error(teste,previsao_ingenua)
rmse_ing=np.sqrt(mean_squared_error(teste,previsao_ingenua))
mae_mod=mean_absolute_error(teste,previsao_modelo)
rmse_mod=np.sqrt(mean_squared_error(teste,previsao_modelo))
metricas=pd.DataFrame({"MAE":[mae_ing,mae_mod],"RMSE":[rmse_ing,rmse_mod]},index=["Referência ingênua","Modelo selecionado"])
display(metricas.round(3))

ax=treino.plot(figsize=(12,5),label="Treinamento")
teste.plot(ax=ax,label="Observado: teste",color="black")
previsao_ingenua.plot(ax=ax,label="Referência ingênua",linestyle="--")
previsao_modelo.plot(ax=ax,label="ARIMA/SARIMA",linestyle="--")
ax.set_title("Previsões no período de teste reservado");ax.legend();plt.show()
display(comparacao.round(2))

## 10. Interpretação e notas metodológicas

MAE expressa o erro absoluto médio na unidade original da série. RMSE penaliza mais intensamente erros de maior magnitude. A comparação deve considerar o horizonte e o período de teste; valores menores indicam menor erro nesse recorte, mas não garantem desempenho futuro.

A seleção de um modelo não deve ser baseada apenas em AIC/BIC nem somente nos erros do teste. O teste foi reservado para avaliação final e não deve orientar a escolha retrospectiva de ordens ou transformações. Se o modelo não superar a referência ingênua, isso deve ser relatado, e não ocultado. A decomposição, os testes de estacionariedade e as ordens ARIMA dependem da série e do intervalo selecionados. Uma transformação logarítmica com retrans formação simples pode introduzir viés na escala original. A sazonalidade deve ser incluída apenas quando sustentada pela periodicidade e pelos padrões observados no treinamento.

### Checklist

- A série mensal, a agregação e o período estão documentados?
- O teste foi reservado antes de escolher transformação e ordens?
- FAC/FACP foram usados como apoio, não como regra mecânica?
- O SARIMA foi incluído somente quando sazonalidade foi justificável?
- Resíduos foram examinados graficamente e por diagnóstico de autocorrelação?
- A comparação inclui uma referência ingênua?
- MAE e RMSE foram calculados somente fora do treinamento e na escala original?
- As limitações de um único corte temporal foram explicitadas?